# Week 17 Optional: Agentic RAG for Data Engineers

## Catalog, Runbooks, and Governance over a Dedicated Bedrock KB

This is an OPTIONAL notebook for the data-engineer persona. It is a self-paced extension of Week 17 for students who manage pipelines, catalogs, and on-call rotations instead of (or in addition to) ML models. Nothing in this notebook depends on the main Week 17 fraud materials; it runs against a SEPARATE Bedrock Knowledge Base provisioned for this purpose.

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Explain agentic RAG for data ops** - when retrieval becomes a tool the pipeline-monitoring agent decides to call, not a one-shot catalog lookup
2. **Query a Bedrock Knowledge Base** with metadata filters so one KB serves three specialist retrievers (catalog, runbook, governance)
3. **Build a multi-retriever pipeline ops supervisor** that extends the Week 16 DE pipeline monitoring pattern by replacing hardcoded lineage and runbook dicts with a queryable KB

## Prerequisites

- Completed Week 16 (Strands agents, agents-as-tools supervisor)
- Completed Week 16 Optional: Data Engineering Agents (pipeline monitoring agent, `PIPELINE_RUNS`, `DATASET_LINEAGE` hardcoded dicts)
- AWS credentials via SageMaker execution role (same as Week 13 / 15 / 16 / 17 main)
- `STRANDS_DE_KNOWLEDGE_BASE_ID` set by your instructor

## The Story

In Week 16 DE optional you built a pipeline monitoring agent with hardcoded `lineage` and `runbooks` dicts. That does not survive real ops:

- Dataset schemas change. Did your hardcoded lineage update? No.
- A new runbook was authored yesterday for a new CDC failure mode. Is it in the agent? No.
- Legal revised the retention policy last week. Does the agent cite the current version? No.

Today we grow those dicts up. They become a real Bedrock KB with three doc types (catalog / runbook / governance), and the pipeline-monitoring agent LEARNS when to consult each corpus via metadata filters.

```mermaid
graph TD
    ONCALL[On-call: 'RUN-001 failed - what do I do?']
    SUP[Pipeline Ops Supervisor]

    PM[Pipeline Monitor<br/>Week 16 DE]
    CAT[CatalogRetriever<br/>doc_type=catalog]
    RB[RunbookRetriever<br/>doc_type=runbook]
    GOV[GovernanceRetriever<br/>doc_type=governance]
    DEC[Incident Decision]

    KB[(Bedrock KB<br/>19 docs<br/>doc_type metadata)]

    ONCALL --> SUP
    SUP --> PM
    SUP --> CAT
    SUP --> RB
    SUP --> GOV
    SUP --> DEC

    CAT --> KB
    RB --> KB
    GOV --> KB

    style CAT fill:#e8f5e9,stroke:#4caf50,stroke-width:3px
    style RB fill:#e8f5e9,stroke:#4caf50,stroke-width:3px
    style GOV fill:#e8f5e9,stroke:#4caf50,stroke-width:3px
    style KB fill:#fff3e0,stroke:#ff9800
```

Three specialists, one KB, metadata filters. Production pattern.

## GPU / Runtime

No GPU needed. All work is API-based through Amazon Bedrock. A CPU SageMaker Studio kernel is sufficient. Allow ~45 minutes for self-paced completion.

# Section 0: Environment Setup

Same stack as Week 17 main (Strands + Bedrock + SageMaker role), pointed at a DIFFERENT knowledge base (`STRANDS_DE_KNOWLEDGE_BASE_ID`) that your instructor provisioned with data-engineering content:

- `catalog/` dataset cards for `customers`, `transactions`, `reporting_tables`, `fraud_detection_model_features`, `customer_360`, `ml_feature_store`, `events_stream`, `user_sessions`
- `runbook/` recovery procedures (schema validation, backfill, stale data, CDC, partition missing, Airflow restart)
- `governance/` policies (PII, retention, SLA definitions, change management, escalation matrix)

Every doc was ingested with a `.metadata.json` sidecar tagging its `doc_type`, which we use for filtered retrieval below.

The setup cell below sets the required env vars BEFORE importing `strands_tools` (critical - the tool reads them at import time), runs three pre-flight probes (LLM access, KB resolves, metadata filter works), and fails loud if anything is missing.

In [ ]:
# =============================================================================
# INSTALL REQUIRED LIBRARIES (same as Week 17 main)
# =============================================================================
# If you already ran the main Week 17 notebook in this kernel, these are
# already installed. The %pip line is idempotent - safe to re-run.

%pip install -q \
    "strands-agents>=1.37,<2" \
    "strands-agents-tools[mem0-memory]>=0.2" \
    "boto3>=1.35" \
    "faiss-cpu>=1.8,<2" \
    "rank_bm25>=0.2.2" \
    "opensearch-py>=2.4" \
    "numpy<2"

print("\nPackages installed.")
print("If this was your first install, RESTART THE KERNEL before running the next cell.")

In [ ]:
# =============================================================================
# IMPORTS + ENV HOTFIX
# =============================================================================
# strands_tools.retrieve reads KNOWLEDGE_BASE_ID at IMPORT TIME. We set it
# BEFORE the strands_tools import so the tool works without a kernel restart.
#
# For this notebook we point strands_tools at the DE KB, not the fraud KB.

import os
import json
import boto3
import sagemaker
from sagemaker import get_execution_role
from importlib.metadata import version as pkg_version


# =============================================================================
# SET strands_tools.retrieve ENVIRONMENT VARIABLES (before import)
# =============================================================================
# Prefer STRANDS_DE_KNOWLEDGE_BASE_ID for this notebook. If unset, the
# instructor's KB id is the fallback so the notebook runs out of the box.

STRANDS_DE_KNOWLEDGE_BASE_ID = (
    os.environ.get("STRANDS_DE_KNOWLEDGE_BASE_ID")
    or "HWLCYXAEGP"  # instructor-provisioned DE KB (Apr 2026)
)
os.environ["STRANDS_DE_KNOWLEDGE_BASE_ID"] = STRANDS_DE_KNOWLEDGE_BASE_ID

# Strands tools read KNOWLEDGE_BASE_ID (not STRANDS_KNOWLEDGE_BASE_ID).
# Point it at the DE KB for this notebook.
os.environ["KNOWLEDGE_BASE_ID"] = STRANDS_DE_KNOWLEDGE_BASE_ID

# Lower score threshold: DE corpus is smaller so broader matches are OK.
os.environ["MIN_SCORE"] = os.environ.get("MIN_SCORE", "0.2")
os.environ["RETRIEVE_ENABLE_METADATA_DEFAULT"] = "true"


# =============================================================================
# STRANDS IMPORTS (after env vars)
# =============================================================================
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import retrieve

for pkg in ["strands-agents", "strands-agents-tools", "boto3"]:
    try:
        print(f"  {pkg:28s} {pkg_version(pkg)}")
    except Exception:
        print(f"  {pkg:28s} (not installed)")


# =============================================================================
# SAGEMAKER SESSION + EXECUTION ROLE (same pattern as Week 15/16/17)
# =============================================================================
sess = sagemaker.Session()
role = get_execution_role()
AWS_REGION = sess.boto_region_name

os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

print(f"\nSageMaker execution role: {role.split('/')[-1]}")
print(f"AWS Region:               {AWS_REGION}")


# =============================================================================
# MODEL CONFIGURATION (same model as Week 15/16/17 main)
# =============================================================================
MODEL_ID       = "us.anthropic.claude-3-haiku-20240307-v1:0"
EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"

llm = BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION)

print(f"\nAgent LLM:       {MODEL_ID}")
print(f"Embedding model: {EMBED_MODEL_ID}")
print(f"DE KB:           {STRANDS_DE_KNOWLEDGE_BASE_ID}")


# =============================================================================
# PRE-FLIGHT PROBES
# =============================================================================
# Probe 1: LLM access
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)
try:
    probe = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "ping"}]}],
        inferenceConfig={"maxTokens": 10, "temperature": 0},
    )
    print(f"\nLLM probe OK:          {probe['output']['message']['content'][0]['text']!r}")
except Exception as e:
    print(f"\nLLM probe FAILED: {e}")
    print(f"Ask your instructor to enable Bedrock access for {MODEL_ID}.")
    raise

# Probe 2: DE Knowledge Base resolves
bedrock_agent = boto3.client("bedrock-agent", region_name=AWS_REGION)
try:
    kb_info = bedrock_agent.get_knowledge_base(knowledgeBaseId=STRANDS_DE_KNOWLEDGE_BASE_ID)
    print(f"KB probe OK:           {STRANDS_DE_KNOWLEDGE_BASE_ID} ({kb_info['knowledgeBase']['name']})")
except Exception as e:
    print(f"KB probe FAILED: {e}")
    print("Ask your instructor for the correct STRANDS_DE_KNOWLEDGE_BASE_ID.")
    raise

# Probe 3: metadata filter returns results (confirms sidecars ingested)
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)
try:
    r = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=STRANDS_DE_KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": "customers dataset SLA"},
        retrievalConfiguration={
            "vectorSearchConfiguration": {
                "numberOfResults": 3,
                "filter": {"equals": {"key": "doc_type", "value": "catalog"}},
            },
        },
    )
    n = len(r.get("retrievalResults", []))
    print(f"Metadata filter probe: {n} chunks matched doc_type=catalog")
    if n == 0:
        raise RuntimeError("0 matches - sidecars may not be ingested.")
except Exception as e:
    print(f"Metadata filter probe FAILED: {e}")
    raise

print("\nEnvironment ready.")

In [ ]:
# =============================================================================
# WEEK 16 DE RECAP - RE-DECLARE PIPELINE RUNS + DATASET LINEAGE + TOOLS
# =============================================================================
# Same data and tools as the Week 16 Data Engineering Agents optional. We
# re-declare inline so this notebook runs without importing anything from
# a prior kernel. In a production codebase this would live in a shared module.

PIPELINE_RUNS = {
    "RUN-001": {
        "pipeline":       "nightly_customers_etl",
        "status":         "FAILED",
        "started_at":     "2026-04-16 02:00:00",
        "failed_at":      "2026-04-16 02:17:34",
        "rows_expected":  125000,
        "rows_processed": 0,
        "error":          "SchemaValidationError: Column 'credit_score' expected INT, got VARCHAR",
        "source":         "postgres://prod-db/customers",
        "target":         "s3://data-lake/warehouse/customers/",
        "owner":          "data-platform-team",
    },
    "RUN-002": {
        "pipeline":       "hourly_transactions_cdc",
        "status":         "SUCCESS",
        "started_at":     "2026-04-16 14:00:00",
        "finished_at":    "2026-04-16 14:03:22",
        "rows_expected":  12000,
        "rows_processed": 12450,
        "error":          None,
        "source":         "postgres://prod-db/transactions",
        "target":         "s3://data-lake/warehouse/transactions/",
        "owner":          "data-platform-team",
    },
    "RUN-042": {
        "pipeline":       "weekly_reporting_aggregation",
        "status":         "COMPLETED_WITH_WARNINGS",
        "started_at":     "2026-04-14 06:00:00",
        "finished_at":    "2026-04-14 06:45:12",
        "rows_expected":  500000,
        "rows_processed": 498712,
        "error":          None,
        "warnings": [
            "3 columns have >5% null values: email, phone, middle_name",
            "Row count 0.26% below expected threshold",
        ],
        "source":         "s3://data-lake/warehouse/*/",
        "target":         "redshift://analytics/reporting_tables",
        "owner":          "analytics-engineering",
    },
}

# Shallow lineage - the KB has the full version as dataset cards
DATASET_LINEAGE = {
    "customers": {
        "upstream":       ["postgres://prod-db/customers"],
        "downstream":     ["reporting_tables", "ml_feature_store", "customer_360"],
        "schema_version": "v3.2",
        "owner":          "data-platform-team",
    },
    "transactions": {
        "upstream":       ["postgres://prod-db/transactions"],
        "downstream":     ["reporting_tables", "fraud_detection_model"],
        "schema_version": "v5.1",
        "owner":          "data-platform-team",
    },
}


@tool
def lookup_pipeline_run(run_id: str) -> str:
    """Look up details of a data pipeline run by its run ID.

    Args:
        run_id: The pipeline run ID to look up (e.g., 'RUN-001').
    """
    run = PIPELINE_RUNS.get(run_id)
    return json.dumps(run, indent=2) if run else f"Pipeline run {run_id} not found."


@tool
def check_dataset_lineage(dataset_name: str) -> str:
    """Retrieve upstream/downstream lineage for a dataset.

    Args:
        dataset_name: The dataset name (e.g., 'customers', 'transactions').
    """
    lin = DATASET_LINEAGE.get(dataset_name)
    return json.dumps(lin, indent=2) if lin else f"Dataset {dataset_name} not found."


print("Week 16 DE recap loaded.")
print(f"  Pipeline runs:   {list(PIPELINE_RUNS.keys())}")
print(f"  Dataset lineage: {list(DATASET_LINEAGE.keys())}")
print(f"  Tools:           lookup_pipeline_run, check_dataset_lineage")

# Section 1: Agentic RAG for Data Ops

## Two Shapes of RAG

In Week 13 you called a Bedrock KB directly with `retrieveAndGenerate` - one-shot Q&A. Perfect for "what does the customers dataset look like?" in a chat UI.

AGENTIC RAG is different. Retrieval is a TOOL the pipeline-monitoring agent can call, skip, or call again based on what the on-call engineer asks.

```mermaid
graph LR
    subgraph "Naive (Week 13)"
        Q1[Query] --> R1[Retrieve top-k]
        R1 --> G1[Generate answer]
    end

    subgraph "Agentic (Week 17 DE)"
        Q2[Query] --> A[Pipeline ops agent]
        A -->|lookup run?| R2A[Run info]
        A -->|consult catalog?| R2B[KB retrieve]
        A -->|consult runbook?| R2C[KB retrieve]
        R2A --> A
        R2B --> A
        R2C --> A
        A --> G2[Compose incident response]
    end

    style A fill:#e8f5e9,stroke:#4caf50,stroke-width:2px
```

## Cost / Benefit

| Shape | Cost | Latency | Best for |
|-------|------|---------|----------|
| **Naive** | 1 LLM call + 1 retrieval | ~1-2s | Single-hop catalog lookup |
| **Agentic** | 3-10x tokens (loop) | ~5-15s | "What do I do about RUN-001" (needs run info + lineage + runbook + policy) |

Published benchmarks put agentic RAG at 3-10x the token cost of naive RAG. The quality lift justifies the spend ONLY when the workload actually needs multi-step reasoning. Incident response is that case.

In [ ]:
# =============================================================================
# DEMO: Naive RAG on a Catalog Query
# =============================================================================
# Week 13 pattern: one retrieve + one LLM call. No agent.

def naive_rag(question: str, top_k: int = 3) -> str:
    """Classic RAG: retrieve top-k chunks, stuff into prompt, generate."""
    resp = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=STRANDS_DE_KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": question},
        retrievalConfiguration={
            "vectorSearchConfiguration": {"numberOfResults": top_k}
        },
    )
    chunks = [r["content"]["text"] for r in resp["retrievalResults"]]
    context = "\n\n---\n\n".join(chunks)

    prompt = (
        "You are a data platform assistant. Answer using ONLY the context below.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    )
    answer = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 400, "temperature": 0},
    )
    return answer["output"]["message"]["content"][0]["text"]


q = "What is the schema and SLA of the customers dataset?"
print("Naive RAG answer:")
print("-" * 70)
print(naive_rag(q))

In [ ]:
# =============================================================================
# DEMO: Agentic RAG with the Plain Strands retrieve Tool
# =============================================================================
# strands_tools.retrieve reads KNOWLEDGE_BASE_ID from env (set in Cell 3).
# It returns a pre-formatted string inside a ToolResult envelope - good for
# the agent to read, but unfiltered: the agent can pull any doc_type.

agentic_rag_agent = Agent(
    model=llm,
    tools=[retrieve],
    system_prompt=(
        "You are a data platform assistant. You have access to a Knowledge "
        "Base of dataset cards, on-call runbooks, and governance docs via "
        "the retrieve tool. When asked a question, call retrieve with a "
        "focused query, read the results, and answer based only on what "
        "the KB returned. If the KB does not cover the question, say so."
    ),
)

print("Agentic RAG answer:")
print("-" * 70)
response = agentic_rag_agent(q)
print(response)
print()
print("Notice: the agent picked its own query wording. But it cannot")
print("constrain doc_type - a schema question might return a runbook chunk.")
print("Section 2 solves that with metadata filters.")

# Section 2: Metadata Filtering = Three Specialists, One KB

## The Problem Section 1 Left Open

The plain `strands_tools.retrieve` tool cannot constrain which `doc_type` it pulls. An agent asking "what is the SLA of customers?" might get a runbook chunk about "SLA breach recovery" instead of the catalog card that DEFINES the SLA.

Real data platforms solve this with METADATA. Every doc in the catalog has attributes - `doc_type`, `owner`, `classification`, `last_updated`. Bedrock Knowledge Bases support metadata filtering at query time.

```mermaid
graph LR
    A[Agent calls retrieve<br/>with filter] --> B[Bedrock KB]
    B --> C[filter: doc_type == catalog]
    C --> D[only catalog chunks<br/>returned]
    D --> A

    style C fill:#e8f5e9,stroke:#4caf50,stroke-width:2px
```

## How Metadata Filtering Works

At INGESTION time, each doc has a sidecar `<filename>.metadata.json`:

```json
{"metadataAttributes": {"doc_type": "catalog", "owner": "data-platform-team"}}
```

At QUERY time, pass a filter to `retrievalConfiguration`:

```python
bedrock_agent_runtime.retrieve(
    knowledgeBaseId=STRANDS_DE_KNOWLEDGE_BASE_ID,
    retrievalQuery={"text": "customers SLA"},
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "numberOfResults": 3,
            "filter": {"equals": {"key": "doc_type", "value": "catalog"}},
        },
    },
)
```

Operators supported on S3 Vectors: `equals`, `notEquals`, `in`, `notIn`, `greaterThan`, `lessThan`, `andAll`, `orAll`. NOT supported on S3 Vectors: `startsWith`, `stringContains` (OpenSearch-only).

## The Multi-Retriever Pattern

Three `doc_type` values give us three specialist retrievers over ONE shared KB:

- `CatalogRetriever` - filter `doc_type=catalog` - schema, SLA, lineage, owner questions
- `RunbookRetriever` - filter `doc_type=runbook` - recovery, on-call, troubleshooting
- `GovernanceRetriever` - filter `doc_type=governance` - PII, retention, escalation

One KB. Three specialists. Metadata filters. Same pattern as the main Week 17 notebook's `PolicyRetriever` + `CaseHistoryRetriever` - just with 3 retrievers over catalog-style content instead of 2 over fraud-policy content.

In [ ]:
# =============================================================================
# DEMO: Filter-Aware Retrieve Helper + Probe All Three doc_types
# =============================================================================
# The plain strands_tools.retrieve does not accept a metadata filter argument,
# so for filtered specialists we call bedrock-agent-runtime.retrieve directly
# from inside a custom @tool wrapper.

def retrieve_with_filter(query: str, doc_type: str, num_results: int = 3) -> list:
    """Retrieve chunks filtered by doc_type. Returns a list of dicts."""
    resp = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=STRANDS_DE_KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={
            "vectorSearchConfiguration": {
                "numberOfResults": num_results,
                "filter": {"equals": {"key": "doc_type", "value": doc_type}},
            },
        },
    )
    out = []
    for r in resp["retrievalResults"]:
        out.append({
            "text":     r["content"]["text"],
            "score":    r.get("score"),
            "source":   r.get("location", {}),
            "metadata": r.get("metadata", {}),
        })
    return out


# Probe: same question, each of the 3 doc_type filters.
# Shows how scores shift based on WHICH corpus we're searching.
q = "What happens when the customers schema validation fails at 2am?"
for doc_type in ["catalog", "runbook", "governance"]:
    hits = retrieve_with_filter(q, doc_type=doc_type, num_results=2)
    print(f"\ndoc_type={doc_type}: {len(hits)} chunks")
    for i, h in enumerate(hits, 1):
        preview = h["text"][:110].replace("\n", " ")
        print(f"  #{i} score={h['score']:.3f}  {preview}...")

print("\nObservation: schema-validation query scores HIGHEST against runbook")
print("(the recovery procedure), LOWER against catalog (dataset cards don't")
print("cover failure modes), and LOWEST against governance (no policy hits).")
print("That is exactly what metadata-filtered retrievers give you for free.")

In [ ]:
# =============================================================================
# DEMO: CatalogRetriever Specialist Agent
# =============================================================================
# A specialist agent has a focused system prompt and a focused tool set.
# We give this one a CUSTOM @tool that always filters on doc_type=catalog.

@tool
def retrieve_catalog(query: str) -> str:
    """Retrieve dataset-card (catalog) chunks from the data platform KB.

    Use for questions about dataset schema, owner, SLA, PII, or lineage.

    Args:
        query: The search query about datasets or their metadata.
    """
    hits = retrieve_with_filter(query, doc_type="catalog", num_results=3)
    return json.dumps({"matches": hits, "count": len(hits)}, indent=2)


catalog_retriever_agent = Agent(
    model=llm,
    tools=[retrieve_catalog],
    system_prompt=(
        "You are a data catalog specialist. Your ONLY job is to answer "
        "questions about dataset schema, ownership, SLA, PII classification, "
        "and lineage by querying the catalog portion of the Knowledge Base. "
        "Rules: "
        "1) Always call retrieve_catalog before answering. "
        "2) Answer ONLY from retrieved content. "
        "3) If the catalog does not cover the dataset, say "
        "'The catalog does not have a card for this dataset.' and stop. "
        "4) Cite the source location(s) returned."
    ),
    callback_handler=None,
)

demo_q = "What is the schema and SLA of the customers dataset?"
print("CatalogRetriever answer:")
print("-" * 70)
print(catalog_retriever_agent(demo_q))

## Lab: Build a RunbookRetriever Specialist (15 min)

### Your Task

Build a specialist RunbookRetriever that follows the same pattern as the CatalogRetriever demo but filtered on `doc_type=runbook`. This retriever will be consumed by the Section 3 ops supervisor, so ship it carefully.

### Steps

1. Create a `retrieve_runbook` `@tool` that calls `retrieve_with_filter(query, doc_type="runbook")` and returns a JSON string with matches + count.
2. Build a `runbook_retriever_agent` Agent that uses your wrapped tool and a strict system prompt scoped to recovery / on-call / troubleshooting.
3. Test on two queries:
   - In-scope: "What do I do when the hourly transactions CDC is stuck?"
   - Out-of-scope: "What is the PII classification of the email column?" (that is a catalog question, not a runbook question)

### Expected Output

- In-scope: a cited answer from the runbook corpus
- Out-of-scope: the agent politely refuses and suggests consulting the catalog

### Stretch (if you finish fast)

Modify `retrieve_runbook` to accept an optional `severity` param and construct a COMPOUND filter with `andAll`:

```python
"filter": {"andAll": [
    {"equals": {"key": "doc_type", "value": "runbook"}},
    {"equals": {"key": "severity", "value": severity}},
]}
```

Our current runbooks have no `severity` attribute, so this will return 0 hits. That is a GOOD thing - it shows the filter is working. Try it, observe the 0-match response, and write a 2-line note on why production runbook corpora benefit from a severity tag.

### Homework

Author a new runbook doc (markdown + sidecar) for a scenario the KB does not currently cover (e.g., "Recovery from Redshift cluster resize mid-write"). Upload to the DE KB's S3 source bucket, run `start_ingestion_job` via boto3, wait for completion, and verify your new runbook is retrievable. Document the end-to-end latency.

In [ ]:
# =============================================================================
# SOLUTION: BUILD A RUNBOOKRETRIEVER SPECIALIST
# =============================================================================
# Approach:
# - Wrap retrieve_with_filter with doc_type="runbook" so the agent can only
#   pull recovery / on-call / troubleshooting docs. Same pattern as the
#   CatalogRetriever demo, different doc_type.
# - System prompt enforces scope: refuse out-of-scope questions politely.

@tool
def retrieve_runbook(query: str) -> str:
    """Retrieve on-call runbook chunks for recovery and troubleshooting.

    Args:
        query: The search query about an ops scenario.
    """
    hits = retrieve_with_filter(query, doc_type="runbook", num_results=3)
    return json.dumps({"matches": hits, "count": len(hits)}, indent=2)


runbook_retriever_agent = Agent(
    model=llm,
    tools=[retrieve_runbook],
    system_prompt=(
        "You are an on-call runbook specialist. Your ONLY job is to answer "
        "recovery, troubleshooting, and on-call procedure questions by "
        "querying the runbook portion of the Knowledge Base. "
        "Rules: "
        "1) Always call retrieve_runbook before answering. "
        "2) Answer ONLY from retrieved content. "
        "3) If the runbook corpus does not cover the scenario, say "
        "'No runbook covers this scenario.' and suggest consulting the "
        "catalog for dataset-level questions. "
        "4) Cite the source location(s) returned."
    ),
    callback_handler=None,
)

# In-scope: recovery question - should retrieve the CDC runbook
in_scope_answer = runbook_retriever_agent(
    "What do I do when the hourly transactions CDC is stuck?"
)
print("In-scope:")
print(in_scope_answer)

# Out-of-scope: catalog question masquerading as ops - should refuse
out_of_scope_answer = runbook_retriever_agent(
    "What is the PII classification of the email column?"
)
print("\nOut-of-scope:")
print(out_of_scope_answer)

# Explanation:
# - Scoping by doc_type is how one KB serves multiple specialist retrievers
#   without cross-contamination. The filter runs at query time so the agent
#   NEVER sees out-of-corpus chunks.
# - Common mistake: forgetting the system-prompt refusal clause. Without it,
#   the agent fabricates a pseudo-answer from the top-N weak matches instead
#   of saying "I don't know."
# - In production, add a second filter (e.g. severity) via andAll so on-call
#   queries can scope to critical-only runbooks during active incidents.

# Section 3: Multi-Retriever Pipeline Ops Supervisor (Main Outcome)

## The Goal

Extend the Week 16 DE pipeline monitoring supervisor so that:

1. The hardcoded `lineage` and `runbooks` dicts from Week 16 DE are REPLACED by three KB-backed specialists: `CatalogRetriever`, `RunbookRetriever`, `GovernanceRetriever`.
2. The supervisor routes: schema/SLA questions to catalog, recovery/ops questions to runbook, policy/PII/retention questions to governance, and full incident investigations through the whole pipeline.

## Supervisor Architecture

```mermaid
graph TD
    USER[On-call engineer query]
    SUP[Pipeline Ops Supervisor<br/>max_iterations=5<br/>max_execution_time=30]

    RPM[run_pipeline_monitor]
    RCAT[run_catalog_retriever]
    RRB[run_runbook_retriever]
    RGOV[run_governance_retriever]
    RDEC[run_incident_decision]

    USER --> SUP
    SUP --> RPM
    SUP --> RCAT
    SUP --> RRB
    SUP --> RGOV
    SUP --> RDEC

    RCAT -->|filter:doc_type=catalog| KB[(Bedrock KB)]
    RRB -->|filter:doc_type=runbook| KB
    RGOV -->|filter:doc_type=governance| KB

    style RCAT fill:#e8f5e9,stroke:#4caf50,stroke-width:3px
    style RRB fill:#e8f5e9,stroke:#4caf50,stroke-width:3px
    style RGOV fill:#e8f5e9,stroke:#4caf50,stroke-width:3px
    style SUP fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
```

## Iteration Budget

`max_iterations=5` + `max_execution_time=30` are the production-required control. Without them a confused supervisor can chain 20 retrievals in a minute. Strands docs call `max_iterations=5` "non-negotiable for production."

## Main Lab

The next code cell declares `run_catalog_retriever`, `run_governance_retriever`, `run_pipeline_monitor`, `run_incident_decision`, and the final supervisor. Then the cell after it tests the supervisor on 3 queries exercising 3 routing paths. The `run_runbook_retriever` tool is YOUR lab output - the safety-net above ensures it exists either way.

In [ ]:
# =============================================================================
# DEMO: Declare Remaining Specialists + Wrap Everything as Tools
# =============================================================================
# Same agents-as-tools pattern as Week 16 DE. Each @tool is a thin wrapper
# that forwards to its specialist agent.

# --- GovernanceRetriever specialist ---
@tool
def retrieve_governance(query: str) -> str:
    """Retrieve data governance chunks - PII, retention, SLA, escalation.

    Args:
        query: Natural-language question about a governance policy.
    """
    hits = retrieve_with_filter(query, doc_type="governance", num_results=3)
    return json.dumps({"matches": hits, "count": len(hits)}, indent=2)


governance_retriever_agent = Agent(
    model=llm,
    tools=[retrieve_governance],
    system_prompt=(
        "You are a data governance specialist. Answer questions about PII, "
        "retention, SLA definitions, change management, and escalation by "
        "querying the governance portion of the Knowledge Base. Cite sources. "
        "If the governance corpus does not cover the question, say so."
    ),
    callback_handler=None,
)


# --- Week 16 DE pipeline monitor + incident decision agents ---
pipeline_monitor_agent = Agent(
    model=llm,
    tools=[lookup_pipeline_run, check_dataset_lineage],
    system_prompt=(
        "You inspect pipeline runs and datasets. Look up the run, check "
        "lineage, produce a factual status report. No recommendations - "
        "that is the decision agent's job."
    ),
    callback_handler=None,
)

incident_decision_agent = Agent(
    model=llm,
    tools=[],
    system_prompt=(
        "You render final on-call verdicts: AUTO_RECOVER, ESCALATE_TO_OWNER, "
        "or PAGE_INCIDENT_COMMANDER. Use the monitor report, catalog context, "
        "runbook context, and governance context you are given."
    ),
    callback_handler=None,
)


# --- Wrap everything as @tool callables ---
@tool
def run_pipeline_monitor(run_id: str) -> str:
    """Inspect a pipeline run and its affected dataset's lineage.

    Args:
        run_id: The pipeline run ID (e.g., 'RUN-001').
    """
    return str(pipeline_monitor_agent(f"Inspect pipeline run {run_id}."))


@tool
def run_catalog_retriever(question: str) -> str:
    """Ask the catalog specialist about dataset schema, SLA, PII, lineage.

    Args:
        question: Natural-language question about a dataset's metadata.
    """
    return str(catalog_retriever_agent(question))


@tool
def run_runbook_retriever(question: str) -> str:
    """Ask the runbook specialist about recovery, on-call, troubleshooting.

    Args:
        question: Natural-language question about an ops scenario.
    """
    return str(runbook_retriever_agent(question))


@tool
def run_governance_retriever(question: str) -> str:
    """Ask the governance specialist about PII, retention, escalation policies.

    Args:
        question: Natural-language question about a governance policy.
    """
    return str(governance_retriever_agent(question))


@tool
def run_incident_decision(monitor: str, catalog: str, runbook: str,
                          governance: str) -> str:
    """Render the on-call verdict from all context.

    Args:
        monitor:    Pipeline monitor report.
        catalog:    Catalog context.
        runbook:    Runbook context.
        governance: Governance context.
    """
    return str(incident_decision_agent(
        f"Monitor: {monitor}\n\nCatalog: {catalog}\n\nRunbook: {runbook}\n\n"
        f"Governance: {governance}\n\n"
        "Render the on-call verdict."
    ))


print("All 5 specialist wrappers declared:")
print("  run_pipeline_monitor, run_catalog_retriever, run_runbook_retriever,")
print("  run_governance_retriever, run_incident_decision")

In [ ]:
# =============================================================================
# DEMO: Pipeline Ops Supervisor + 3 Test Queries
# =============================================================================

ops_supervisor = Agent(
    model=llm,
    tools=[
        run_pipeline_monitor,
        run_catalog_retriever,
        run_runbook_retriever,
        run_governance_retriever,
        run_incident_decision,
    ],
    system_prompt=(
        "You are the pipeline ops supervisor. Route queries by intent: "
        "  - PURE schema/SLA/PII/lineage questions (no run ID) -> "
        "    run_catalog_retriever ONLY and return its answer. "
        "  - PURE recovery/troubleshooting questions (no run ID) -> "
        "    run_runbook_retriever ONLY and return its answer. "
        "  - PURE governance/retention/escalation questions -> "
        "    run_governance_retriever ONLY and return its answer. "
        "  - FULL INCIDENT (a RUN-NNN is mentioned) -> call "
        "    run_pipeline_monitor first, then run_catalog_retriever AND "
        "    run_runbook_retriever AND run_governance_retriever for context, "
        "    then run_incident_decision with all four context strings. "
        "Do NOT call specialists you don't need. Conserve tool calls."
    ),
    max_iterations=5,
    max_execution_time=30,
)


# Query 1: catalog-only
print("=" * 70)
print("Q1 (catalog-only): 'What is the SLA of reporting_tables?'")
print("=" * 70)
print(ops_supervisor("What is the SLA of reporting_tables?"))
print()

# Query 2: runbook-only
print("=" * 70)
print("Q2 (runbook-only): 'How do we recover from a partition missing error?'")
print("=" * 70)
print(ops_supervisor("How do we recover from a partition missing error on hourly transactions CDC?"))
print()

# Query 3: full incident
print("=" * 70)
print("Q3 (full incident): 'RUN-001 failed at 2:17 AM, what do I do?'")
print("=" * 70)
print(ops_supervisor("RUN-001 failed at 2:17 AM. Render a verdict."))

# Summary: What You Built

## Key Takeaways

### Agentic RAG for Data Ops
- Retrieval is a tool the pipeline ops agent calls when needed, not a pipeline step.
- Cost: 3-10x naive RAG per query. Budget with `max_iterations=5` and `max_execution_time=30`.

### Metadata-Filtered Multi-Retriever Pattern
- One Bedrock KB, three specialists distinguished by `doc_type` metadata filter.
- `.metadata.json` sidecars at ingestion + `filter: {"equals": {"key": ..., "value": ...}}` at retrieval = production pattern.
- S3 Vectors supports `equals`, `notEquals`, `in`, `notIn`, `andAll`, `orAll` - NOT `startsWith` / `stringContains`.
- Custom `@tool` wrappers (not the plain `strands_tools.retrieve`) because filters go through `bedrock-agent-runtime.retrieve` directly.

### Pipeline Ops Supervisor
- Same agents-as-tools pattern as Week 16 DE, now with 3 KB-backed specialists + pipeline monitor + decision agent.
- The hardcoded `lineage` and `runbooks` dicts from Week 16 DE are retired. The KB is the source of truth.

## Homework

### Homework 1: Author and Ingest a New Runbook
Write a new runbook doc + sidecar for a scenario not currently in the KB (e.g., "Redshift cluster resize mid-write recovery"). Upload to the DE KB S3 bucket, run `start_ingestion_job`, verify it is retrievable. Document end-to-end latency.

### Homework 2: Scheduled Re-ingestion (Bridges to Week 21-22)
Your catalog changes daily. The KB cannot be static. Sketch an Airflow DAG that:
- Runs nightly at 03:00 UTC
- Diffs the S3 source prefix against the last ingestion job
- Triggers `start_ingestion_job` if there are new or changed docs
- Alerts on ingestion failures

You don't need to IMPLEMENT it - just sketch the DAG structure in pseudocode.

### Homework 3: Audit-Ready Citations (Bridges to Week 23)
Every retrieve result includes a `location` field with the S3 URI. Modify one of your specialists to ALWAYS append a "Sources: ..." footer to its answer, citing the exact URIs. Why does this matter for GDPR / SOX audits? Write one paragraph.

## Looking Ahead

- **Week 18** (if you continue with the fraud cohort) takes the retrieval pipeline apart: chunking strategies, reranking with Cohere Rerank 3.5, RAG evaluation with RAGAS. Every technique applies directly to this DE KB.
- **Weeks 19-20** add MLOps to these exact agents: DVC versioning your KB docs, MLflow tracking which config wins, Langfuse for online observability.
- **Weeks 21-22** (Airflow) is where Homework 2's DAG goes into production.
- **Week 23** (Ethics) is where the `location` citations become audit evidence.

## Great Work

You just replaced hardcoded ops dicts with a queryable, auditable, citation-emitting knowledge base. That is the foundation of every modern data platform's AI layer.